# Step 1: Get Features From Multiple Datasets
- Using [pybiber](https://pypi.org/project/pybiber/)

In [ ]:
# HuggingFace Login
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)

from transformers import logging
logging.set_verbosity_error()

In [ ]:
import pybiber as pb
import polars as pl
import os
import numpy as np
import random
import torch
from scipy.stats import zscore
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import gc

In [ ]:
DEVICE = 0 if torch.cuda.is_available() else -1
FILE_PATH = 'getText/datasetsPrep'
OUTPUT_DIR = 'biberOutputs'
SAMPLE_SIZE = 2
batch_idx = 0

In [ ]:
# Some models excluded to prevent overemphasis on some architectures (in the averaging steps). (2 lightweight NLI models + 1 medium-length-text optimized model + 1 high-capacity zero-shot model)
ZERO_SHOT_MODELS = [
    "cross-encoder/nli-MiniLM2-L6-H768",
    "typeform/distilbert-base-uncased-mnli",
    "tasksource/deberta-small-long-nli", # Optimized for long texts.
    "MoritzLaurer/deberta-v3-base-zeroshot-v1"
    # "MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli-ling-binary", Excluded as it is designed for binary NLI and could underperform in our multi-label scenario.
    # "MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33", Excluded as it is trained on binary datasets and also as even on the model card it states that the model, while more efficient than its larger sisters, does not perform as well as them.
    # "cmarkea/distilcamembert-base-nli", Excluded as it is for the French language and we are focusing only on English texts.
    # "MoritzLaurer/xtremedistil-l6-h256-zeroshot-v1.1-all-33", Excluded as it is nearly the same as the MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33 model. 
]

In [ ]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [ ]:
# Load all data.
list_of_dfs = []
for folder in os.listdir(f'./{FILE_PATH}'):
    if os.path.isdir(f'./{FILE_PATH}/{folder}'):
        for file in os.listdir(f'./{FILE_PATH}/{folder}'):
            if file.endswith(".csv"):
                temp_file_path = f'./{FILE_PATH}/{folder}/{file}'
                temp_tag = file.replace('.csv', '')
                temp_df = pl.read_csv(temp_file_path)
                temp_df = (
                    temp_df
                    .with_row_index("index_num") # , offset=1) if you want to start index from 1
                    .with_columns(
                        (pl.lit(temp_tag) + "_" + pl.col("index_num").cast(pl.Utf8)).alias("doc_id")
                    )).select(['text', 'doc_id'])
                list_of_dfs.append(temp_df)

combined = pl.concat(list_of_dfs, how="vertical")
assert combined.select(pl.col("doc_id").n_unique()).item() == combined.height, "There should be no duplicates in the dataset."

In [ ]:
# Remove invalid data.
temp_df = combined.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
print(temp_df.group_by("tag").len())
print(f"Total Texts Before Empty String Removal: {len(temp_df)}")

temp_df = combined.with_columns(pl.col("text").str.strip_chars().alias("text")).filter(pl.col("text").is_not_null() & (pl.col("text") != ""))

temp_df = temp_df.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
print(temp_df.group_by("tag").len())
print(f"Total Texts After Empty String Removal: {len(temp_df)}")

In [ ]:
# Randomly sample from dataframe.
temp_df = temp_df.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
temp_df = temp_df.to_pandas()
temp_df = (temp_df.groupby("tag")).apply(lambda x: x.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE + batch_idx)) ; batch_idx += 1 # Increase batch_idx every time to ensure new random samples.
temp_df = pl.from_pandas(temp_df)


In [ ]:
# Set up df for use.
df = pl.DataFrame({
    "doc_id": temp_df['doc_id'].to_list(),
    "text": temp_df['text'].to_list()
})

# Light preprocessing to strip extra whitespace.
df = df.with_columns(
    pl.col("text")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .str.replace_all(r"^\s*-\s*", "") # Remove dashes at the beginning of texts.
    .str.replace_all(r"^\s*\d+\.\s*", "") # Remove numbers in 1., 2., 3. format at the beginning of the text. 
)

pybiber_pipeline = pb.PybiberPipeline(model="en_core_web_sm")
features, tokens = pybiber_pipeline.run(df, return_tokens=True)
features = features.with_columns(pl.col("doc_id").str.split("_").list.get(0).alias("category"))
# Full feature list can be found here: https://browndw.github.io/pybiber/feature-categories.html
print(f" -------- Features-------- ")
print(features)

# Statistical analysis and visualization
analyzer = pb.BiberAnalyzer(features, id_column='category')

# Multi-Dimensional Analysis - see https://browndw.github.io/pybiber/biber-analyzer.html#comparison-with-bibers-original-dimensions for factor mapping
# Explanation of the factor mapping to dimensions can be found here: https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html
'''
Factor 1: Involved vs. Informational Production (negative to positive)
Factor 2: Narrative vs. Non-narrative Concerns (negative to positive)
Factor 3: Explicit vs. Situation-dependent Reference (negative to positive)
Factor 4: Overt Expression of Persuasion (negative to positive)
Factor 5: Abstract vs. Non-abstract Information (negative to positive)
Factor 6: On-line Informational Elaboration (negative to positive)
'''

analyzer.mda_biber()
print(f" -------- MDA Summary -------- ")
print(analyzer.mda_summary)
print(f" -------- MDA Loadings -------- ")
print(analyzer.mda_loadings)
print(f" -------- MDA Dimension Scores -------- ")
print(analyzer.mda_dim_scores)
print(f" -------- MDA Group Means -------- ") # Simple mean calculation (nothing fancy).
print(analyzer.mda_group_means)

In [ ]:
os.makedirs(f"./{OUTPUT_DIR}/", exist_ok=True)

def flatten_for_csv(df):
    # Work on a copy to avoid modifying original.
    df_flat = df.clone()
    for c, dtype in zip(df_flat.columns, df_flat.dtypes):
        if dtype == pl.List:
            # Join list elements into string with commas.
            df_flat = df_flat.with_columns(
                pl.Series(df_flat[c].name, [",".join(map(str, x)) if x is not None else "" for x in df_flat[c]])
            )
        elif dtype == pl.Struct:
            # Convert struct to string representation.
            df_flat = df_flat.with_columns(
                pl.Series(df_flat[c].name, df_flat[c].cast(pl.Utf8))
            )
    return df_flat

# Get Z-Scores from Biber analysis.
biber_dimensions = (analyzer.mda_dim_scores).to_pandas()
biber_dimensions = biber_dimensions.drop(columns=['factor_7']) # Remove factor 7 as it is not used or defined in Biber's original 6 dimensions.
# Get factor columns.
factor_cols = [c for c in biber_dimensions.columns if c.startswith("factor")]
biber_dimensions[factor_cols] = biber_dimensions[factor_cols].apply(zscore)
for c in factor_cols:
    biber_dimensions[f"{c}_label"] = biber_dimensions[c] > 0
biber_dimensions = pl.from_pandas(biber_dimensions)
print(biber_dimensions)

In [ ]:
# Write CSV files.
flatten_for_csv(biber_dimensions).write_csv(f"./{OUTPUT_DIR}/mda_dim_scores{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)
flatten_for_csv(analyzer.mda_summary).write_csv(f"./{OUTPUT_DIR}/mda_summary{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)
flatten_for_csv(analyzer.mda_loadings).write_csv(f"./{OUTPUT_DIR}/mda_loadings{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)
flatten_for_csv(analyzer.mda_group_means).write_csv(f"./{OUTPUT_DIR}/mda_group_means{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)

# Write JSON files.
analyzer.mda_summary.write_json(f"./{OUTPUT_DIR}/mda_summary{RANDOM_STATE + batch_idx - 1}.json")
analyzer.mda_loadings.write_json(f"./{OUTPUT_DIR}/mda_loadings{RANDOM_STATE + batch_idx - 1}.json")
analyzer.mda_dim_scores.write_json(f"./{OUTPUT_DIR}/mda_dim_scores{RANDOM_STATE + batch_idx - 1}.json")
analyzer.mda_group_means.write_json(f"./{OUTPUT_DIR}/mda_group_means{RANDOM_STATE + batch_idx - 1}.json")

In [ ]:
# Exact mapping taken from https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html which has extracted the same from Biber and Conrad's Variation in English (https://doi.org/10.4324/9781315840888)
BIBER_LABEL_MAP = {
    "factor_1": {"informational": -1, "involved": 1},
    "factor_2": {"non-narrative": -1, "narrative": 1},
    "factor_3": {"situation-dependent": -1, "explicit": 1},
    "factor_4": {"non-persuasive": -1, "persuasive": 1},
    "factor_5": {"non-abstract": -1, "abstract": 1},
    "factor_6": {"compressed": -1, "elaborated": 1}
}

# Using different prompt templates increases robustness. 
TEMPLATES = [
    "This text is {}.",
    "This text is written in a {} style.",
    "The writing style of this text is {}.",
    "This text can be described as {}."
]

In [ ]:
temp_df = temp_df.to_pandas()
texts = temp_df['text'].values.tolist()
doc_ids = temp_df['doc_id'].values.tolist()

In [ ]:
assert len(texts) == len(doc_ids), "texts and doc_ids are not of the same length."

In [ ]:
rows = []
for model_name in ZERO_SHOT_MODELS:
    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        device=-1
    )
    for i in range(len(texts)):
        temp_factors = {'model_name': model_name, 'doc_id': doc_ids[i]}
        for factor, description in BIBER_LABEL_MAP.items():
            final_dimension_score = 0
            for template in TEMPLATES:
                outputs = classifier(
                    texts[i],
                    candidate_labels=list(description.keys()),
                    hypothesis_template=template,
                    batch_size=8,
                    multi_label = True
                )
                for label, score in zip(outputs['labels'], outputs['scores']):
                    final_dimension_score += description[label] * score
            final_dimension_score = final_dimension_score / len(TEMPLATES)
            temp_factors[factor] = final_dimension_score
        rows.append(temp_factors)
    # Free memory.
    del classifier
    torch.cuda.empty_cache()
    gc.collect()

    # break

df = pd.DataFrame(rows)
mean_scores = df.drop(columns='model_name').groupby("doc_id").mean().reset_index()

In [ ]:
mean_scores

In [ ]:
df